# 📊 Разведочный анализ датасета RuSentiment

Прежде чем обучать модели — нужно хорошо понять данные. Этот ноутбук отвечает на ключевые вопросы:

- Сколько примеров в датасете и как они распределены по классам?
- Насколько сбалансирован датасет?
- Какова типичная длина текста — укладываемся ли в `max_length=128`?
- Как выглядят тексты каждого класса?

**Классы тональности:**
| label_id | Класс | Описание |
|----------|-------|----------|
| 0 | negative | Негативные отзывы и высказывания |
| 1 | neutral  | Нейтральные, информационные тексты |
| 2 | positive | Позитивные отзывы и высказывания |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

# Стиль графиков
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Цветовая схема: красный / серый / зелёный — интуитивно понятно
COLORS = {0: '#e74c3c', 1: '#95a5a6', 2: '#2ecc71'}
LABEL_NAMES = {0: 'negative', 1: 'neutral', 2: 'positive'}

print('Готово к анализу!')

## 1. Загрузка данных

Данные уже разбиты на три сплита скриптом `preprocessing.py` в соотношении **80 / 10 / 10** со стратификацией по классам. Загружаем все три и объединяем для общего анализа.

In [ ]:
train_df = pd.read_csv('../data/processed/train.csv')
val_df   = pd.read_csv('../data/processed/val.csv')
test_df  = pd.read_csv('../data/processed/test.csv')

# Объединяем для общего анализа, добавляем колонку сплита
train_df['split'] = 'train'
val_df['split']   = 'val'
test_df['split']  = 'test'
df = pd.concat([train_df, val_df, test_df], ignore_index=True)

print('Размеры сплитов:')
print(f'  train : {len(train_df):>7,}  ({len(train_df)/len(df)*100:.1f}%)')
print(f'  val   : {len(val_df):>7,}  ({len(val_df)/len(df)*100:.1f}%)')
print(f'  test  : {len(test_df):>7,}  ({len(test_df)/len(df)*100:.1f}%)')
print(f'  итого : {len(df):>7,}')
print()
df[['text', 'label_id']].head()

## 2. Распределение классов

Один из первых вопросов при работе с задачей классификации — **насколько сбалансирован датасет?**

Сильный дисбаланс классов — серьёзная проблема: модель может просто научиться предсказывать доминирующий класс и получить высокую accuracy, ничего не поняв. В таких случаях нужно либо взвешивать потери (class weights), либо делать oversampling.

Посмотрим, что у нас.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = df['label_id'].value_counts().sort_index()

# --- Столбчатая диаграмма ---
bars = axes[0].bar(
    [LABEL_NAMES[i] for i in counts.index],
    counts.values,
    color=[COLORS[i] for i in counts.index],
    edgecolor='white', linewidth=1.5, width=0.5
)
for bar, val in zip(bars, counts.values):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 500,
        f'{val:,}', ha='center', va='bottom', fontsize=11, fontweight='bold'
    )
axes[0].set_title('Количество примеров по классам', fontsize=13, pad=12)
axes[0].set_ylabel('Количество текстов')
axes[0].set_ylim(0, counts.max() * 1.15)

# --- Круговая диаграмма ---
axes[1].pie(
    counts.values,
    labels=[LABEL_NAMES[i] for i in counts.index],
    colors=[COLORS[i] for i in counts.index],
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    textprops={'fontsize': 12}
)
axes[1].set_title('Доля классов в датасете', fontsize=13, pad=12)

plt.suptitle('RuSentiment — распределение классов тональности', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

**Вывод:** датасет практически идеально сбалансирован — каждый класс занимает ~33.3%. Это отличная новость: не нужно применять специальные техники борьбы с дисбалансом, а метрика accuracy будет информативной наравне с F1.

## 3. Длины текстов

Трансформерные модели имеют ограничение на длину входной последовательности. В нашем случае `max_length=128` токенов. Токен — это не всегда слово (например, «необычный» может разбиться на несколько токенов), но количество слов даёт хорошее приближение.

Важно понять: **какой процент текстов будет обрезан?** Обрезка ведёт к потере информации, что может снизить качество модели на длинных текстах.

In [ ]:
df['char_len'] = df['text'].str.len()
df['word_len'] = df['text'].str.split().str.len()

print('Статистика длин текстов:')
stats = df.groupby(df['label_id'].map(LABEL_NAMES))[['word_len', 'char_len']].describe().round(1)
print(stats[['word_len', 'char_len']].to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label_id in [0, 1, 2]:
    subset = df[df['label_id'] == label_id]
    axes[0].hist(
        subset['word_len'].clip(upper=150), bins=50,
        alpha=0.55, color=COLORS[label_id], label=LABEL_NAMES[label_id]
    )
    axes[1].hist(
        subset['char_len'].clip(upper=800), bins=50,
        alpha=0.55, color=COLORS[label_id], label=LABEL_NAMES[label_id]
    )

# Линия max_length
axes[0].axvline(x=128, color='black', linestyle='--', linewidth=1.5, label='max_length = 128')
axes[0].set_title('Длина текстов в словах', fontsize=13)
axes[0].set_xlabel('Количество слов')
axes[0].set_ylabel('Частота')
axes[0].legend()

axes[1].set_title('Длина текстов в символах', fontsize=13)
axes[1].set_xlabel('Количество символов')
axes[1].set_ylabel('Частота')
axes[1].legend()

plt.suptitle('Распределение длин текстов по классам', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/eda_text_lengths.png', dpi=150, bbox_inches='tight')
plt.show()

long_pct = (df['word_len'] > 128).mean() * 100
long_cnt = (df['word_len'] > 128).sum()
print(f'Текстов длиннее 128 слов: {long_cnt:,} ({long_pct:.1f}%) — будут обрезаны токенизатором')
print(f'Медианная длина: {df["word_len"].median():.0f} слов')

**Вывод:** большинство текстов укладываются в 128 слов — ограничение `max_length=128` покрывает основную массу датасета. Длинные тексты будут обрезаны, но их доля невелика, поэтому потери информации минимальны. При желании можно увеличить `max_length=256`, но это вдвое увеличит потребление памяти GPU.

## 4. Проверка стратификации по сплитам

При разбиении на train/val/test мы использовали **стратифицированное разбиение** — это гарантирует, что распределение классов одинаково во всех трёх частях. Убедимся, что это действительно так.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
split_data = [('train', train_df), ('val', val_df), ('test', test_df)]

for ax, (name, split_df) in zip(axes, split_data):
    counts = split_df['label_id'].value_counts().sort_index()
    bars = ax.bar(
        [LABEL_NAMES[i] for i in counts.index],
        counts.values,
        color=[COLORS[i] for i in counts.index],
        edgecolor='white', linewidth=1.5, width=0.5
    )
    for bar, val in zip(bars, counts.values):
        pct = val / len(split_df) * 100
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 50,
            f'{val:,}\n({pct:.1f}%)',
            ha='center', va='bottom', fontsize=9
        )
    ax.set_title(f'{name}  ({len(split_df):,} примеров)', fontsize=12)
    ax.set_ylabel('Количество текстов')
    ax.set_ylim(0, counts.max() * 1.25)

plt.suptitle('Распределение классов в каждом сплите — проверка стратификации', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/eda_splits.png', dpi=150, bbox_inches='tight')
plt.show()

print('Доли классов по сплитам:')
for name, split_df in split_data:
    pcts = split_df['label_id'].value_counts(normalize=True).sort_index() * 100
    print(f'  {name}: ' + '  |  '.join([f'{LABEL_NAMES[i]}: {pcts[i]:.1f}%' for i in range(3)]))

**Вывод:** стратификация сработала корректно — во всех трёх сплитах классы представлены равномерно (~33% каждый). Это важно для достоверной оценки: val и test репрезентативно отражают реальное распределение данных.

## 5. Примеры текстов по классам

Количественный анализ — это хорошо, но не менее важно **почитать сами тексты**. Это помогает понять:
- Насколько очевидна тональность?
- Есть ли шумные или неоднозначные примеры?
- Какой стиль текста преобладает?

In [ ]:
pd.set_option('display.max_colwidth', 300)

for label_id in [0, 1, 2]:
    label_color = {'negative': '🔴', 'neutral': '⚪', 'positive': '🟢'}
    name = LABEL_NAMES[label_id]
    print(f'{label_color[name]} Класс: {name.upper()} (label_id={label_id})')
    print('-' * 70)
    samples = df[df['label_id'] == label_id]['text'].sample(3, random_state=42)
    for i, text in enumerate(samples, 1):
        # Обрезаем очень длинные тексты для читаемости
        display_text = text[:250] + '...' if len(text) > 250 else text
        print(f'{i}. {display_text}')
    print()

## 6. Итоговая сводка

Собираем все ключевые цифры в одном месте.

In [ ]:
print('=' * 55)
print('  СВОДКА ПО ДАТАСЕТУ RuSentiment')
print('=' * 55)
print(f'  Всего примеров      : {len(df):,}')
print(f'  Обучающая выборка   : {len(train_df):,}  (80%)')
print(f'  Валидационная       : {len(val_df):,}   (10%)')
print(f'  Тестовая            : {len(test_df):,}   (10%)')
print()
print('  Распределение классов (сбалансировано):')
for label_id in [0, 1, 2]:
    count = (df['label_id'] == label_id).sum()
    print(f'    {LABEL_NAMES[label_id]:10}: {count:,}  ({count/len(df)*100:.1f}%)')
print()
print('  Длина текстов:')
print(f'    Среднее  : {df["word_len"].mean():.1f} слов')
print(f'    Медиана  : {df["word_len"].median():.0f} слов')
print(f'    Максимум : {df["word_len"].max()} слов')
print()
long_pct = (df['word_len'] > 128).mean() * 100
print(f'  Обрезается при max_length=128: {long_pct:.1f}% текстов')
print()
print('  Выводы для моделирования:')
print('    - Датасет сбалансирован, взвешивание классов не требуется')
print('    - max_length=128 покрывает большинство текстов')
print('    - Метрики accuracy и F1 будут одинаково информативны')
print('=' * 55)